# Format coverage

`corpus-prep` ships with **nine** formats out of the box:

| Format | MIME type | Parser | Notes |
|---|---|---|---|
| Plain text | `text/plain` | `PlainTextParser` | UTF-8 with Latin-1 fallback |
| Markdown | `text/markdown` | `MarkdownParser` | preserves structure |
| CSV | `text/csv` | `CSVParser` | one row per line, `col: val` pairs joined by `&#124;` |
| JSON | `application/json` | `JSONParser` | indented, `ensure_ascii=False` |
| HTML | `text/html` | `HTMLParser` (Trafilatura) | drops boilerplate |
| PDF (native) | `application/pdf` | `PDFNativeParser` (PyMuPDF) | flags `needs_ocr` for scans |
| DOCX | `.../wordprocessingml.document` | `DoclingParser` | IBM Docling |
| PPTX | `.../presentationml.presentation` | `DoclingParser` | IBM Docling |
| Images | `image/png/jpeg/tiff` | `DoclingParser` | OCR via Docling backend |

MIME detection uses **Magika** (Google) and trumps file extensions, so a `.txt` containing JSON is still parsed as JSON.

This notebook builds a synthetic mini-corpus covering the always-on formats and runs the full pipeline against it.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

## 1. Build a tiny multi-format input directory

In [2]:
import json
import shutil
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
in_dir = tmp / "in"
in_dir.mkdir()

(in_dir / "news.txt").write_text(
    "Modelos de linguagem treinados em corpora de larga escala demonstram "
    "comportamentos emergentes a partir de certo numero de parametros. "
    "Pesquisadores em laboratorios de ponta tem demonstrado que a capacidade "
    "de raciocinio em poucos exemplos melhora suavemente com a escala."
)

(in_dir / "recipe.md").write_text(
    "# Baiao de dois\n\n"
    "Prato tradicional do **sertao nordestino** combinando arroz, feijao "
    "de corda, queijo coalho e carne seca. Receita base: cozinhar o feijao, "
    "refogar com cebola e alho, adicionar arroz, completar com agua e "
    "cozinhar ate secar."
)

(in_dir / "city_pop.csv").write_text(
    "municipio,populacao,regiao\n"
    "Teresina,868000,nordeste\n"
    "Parnaiba,153000,nordeste\n"
    "Picos,78000,nordeste\n"
    "Sao Raimundo Nonato,32000,nordeste\n"
    "Joao Costa,3500,nordeste\n"
)

(in_dir / "meta.json").write_text(
    json.dumps(
        {
            "titulo": "Edital 002/25 - Processo Seletivo",
            "orgao": "Prefeitura Municipal de Joao Costa",
            "resumo": (
                "Convocacao de candidatos para etapa de avaliacao de "
                "titulos do processo seletivo simplificado para "
                "contratacao temporaria de profissionais da educacao."
            ),
        },
        ensure_ascii=False,
    )
)

(in_dir / "page.html").write_text(
    "<html><body>"
    "<nav>menu | inicio | servicos | contato</nav>"
    "<article><h1>Caatinga</h1>"
    "<p>O bioma da caatinga ocupa parte significativa do interior "
    "nordestino e abriga especies adaptadas a longos periodos de estiagem. "
    "A vegetacao predominante inclui arvores de pequeno porte, cactos e "
    "plantas com mecanismos de armazenamento de agua bem desenvolvidos.</p>"
    "</article>"
    "<footer>Copyright 2026</footer>"
    "</body></html>"
)

for f in sorted(in_dir.iterdir()):
    print(f"{f.name:20s}  {f.stat().st_size:>6} bytes")

city_pop.csv             158 bytes
meta.json                259 bytes
news.txt                 272 bytes
page.html                406 bytes
recipe.md                240 bytes


## 2. Run the pipeline

We disable the language filter so this demo doesn't depend on the GlotLID model file. In production every output Document carries a language tag.

In [3]:
from corpus_prep.pipeline import PipelineConfig, run_pipeline

out_dir = tmp / "out"
config = PipelineConfig(
    input_dir=in_dir,
    output_dir=out_dir,
    enable_filter=False,
)
report = run_pipeline(config)
report

Output()

RunReport(input_files=5, pre_dedup_kept=5, parsed=5, parse_failures=[], filter_passed=0, filter_rejected={}, post_dedup_kept=5, post_dedup_removed=0, shards_written=1, total_chars_written=1346, duration_seconds=0.30256853199989564)

## 3. Inspect the resulting Parquet

In [4]:
import duckdb

glob = str(out_dir / "shard-*.parquet")
con = duckdb.connect()

print("Documents per parser:")
print(
    con.execute(
        f"SELECT parser, COUNT(*) AS n, SUM(char_count) AS total_chars "
        f"FROM '{glob}' GROUP BY parser ORDER BY n DESC"
    ).fetchdf()
)

Documents per parser:
      parser  n  total_chars
0   markdown  1        240.0
1        csv  1        300.0
2       json  1        264.0
3  plaintext  1        272.0
4       html  1        270.0


In [5]:
print("\nDocument previews:")
rows = con.execute(
    f"SELECT source_path, mime, parser, SUBSTR(text, 1, 100) AS preview "
    f"FROM '{glob}' ORDER BY source_path"
).fetchall()
for source_path, mime, parser, preview in rows:
    print(f"\n--- {source_path} ({mime} via {parser}) ---")
    print(preview)


Document previews:

--- city_pop.csv (text/csv via csv) ---
municipio: Teresina | populacao: 868000 | regiao: nordeste
municipio: Parnaiba | populacao: 153000 |

--- meta.json (application/json via json) ---
{
 "titulo": "Edital 002/25 - Processo Seletivo",
 "orgao": "Prefeitura Municipal de Joao Costa",
 "

--- news.txt (text/plain via plaintext) ---
Modelos de linguagem treinados em corpora de larga escala demonstram comportamentos emergentes a par

--- page.html (text/html via html) ---
Caatinga
O bioma da caatinga ocupa parte significativa do interior nordestino e abriga especies adap

--- recipe.md (text/markdown via markdown) ---
# Baiao de dois

Prato tradicional do **sertao nordestino** combinando arroz, feijao de corda, queij


## 4. Cleanup

In [6]:
shutil.rmtree(tmp)
print("Cleaned up", tmp)

Cleaned up /tmp/tmpkstffumv


## Notes on PDF / DOCX / PPTX / images

The same pipeline handles those formats — they're just heavier to demo:

- **PDF (native)**: drop a real PDF into `in_dir` and `PDFNativeParser` extracts the text layer instantly via PyMuPDF. If the PDF is a scan (no text layer), the parser sets `metadata.needs_ocr=true`.
- **PDF (scanned), DOCX, PPTX, images**: handled by `DoclingParser`. Docling lazily downloads its layout / OCR models on first use (~hundreds of MB), so we kept it out of this notebook to keep execution fast.

To exercise Docling end-to-end, drop a real `.pdf` / `.docx` / `.png` into `in_dir` and re-run cell 2.